# Tìm tham số controller (giai đoạn 1)

Tối ưu ~10 tham số của policy tham chiếu bằng CMA-ES (gói `cma`, nếu thiếu thì random search).

* **Chung**: một bộ cho mọi tòa nhà → nộp được ngay.
* **Riêng** (`MODE = "per_scenario"`): bộ tốt nhất cho từng kịch bản → "thầy" cho giai đoạn 2. Đắt hơn nhiều.

Đánh giá nhanh bằng episode ngắn trên một nhóm kịch bản, rồi kiểm tra bộ tốt nhất trên các kịch bản **không dùng để tune**.
Mọi lần chạy được cache trong `results/search/`, ngắt giữa chừng chạy lại sẽ tiếp tục.

In [ ]:
import os, sys, json, time
from pathlib import Path

# Find the repo (the folder holding sbx/) and the starter kit.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sbx").is_dir())
sys.path.insert(0, str(REPO))
# os.environ["SMART_BUILDINGS_DATA_ROOT"] = "/path/to/starter-kit"   # if not next to the repo
from sbx import paths
DATA_ROOT = paths.setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sbx import evaluation as ev, candidates as cand

WORKERS = int(os.getenv("SB_WORKERS", 0)) or os.cpu_count()   # set to your physical core count
print("repo:", REPO, "\ndata:", DATA_ROOT, "\nworkers:", WORKERS)

In [ ]:
# name: (low, high, kind)   kind: lin | log | int
SPACE = {
    "KP":                  (0.04, 0.20, "log"),
    "KI":                  (0.0015, 0.012, "log"),
    "PRESTART_H":          (1, 4, "int"),
    "HOLD_STEPS":          (48, 576, "log"),
    "NIGHT_HEAT_DROP_K":   (0.0, 3.0, "lin"),
    "MILD_SHIFT_K":        (0.0, 1.0, "lin"),
    "MILD_BELOW_C":        (5.0, 16.0, "lin"),
    "EXTREME_BELOW_C":     (-25.0, -5.0, "lin"),
    "EXTREME_PRESTART_H":  (2, 6, "int"),
    "COLD_SAT":            (-1.0, -0.3, "lin"),
    "MIN_DAMPER":          (0.1, 0.3, "lin"),
}
REF_VALUES = {k: getattr(cand.REGISTRY["ref"][0], k) for k in SPACE}
print(REF_VALUES)

def decode(x):
    out = {}
    for (k, (lo, hi, kind)), v in zip(SPACE.items(), np.clip(x, 0, 1)):
        if kind == "log":
            val = float(np.exp(np.log(lo) + v * (np.log(hi) - np.log(lo))))
        else:
            val = lo + v * (hi - lo)
        out[k] = int(round(val)) if kind == "int" or k == "HOLD_STEPS" else round(val, 5)
    return out

def encode(values):
    x = []
    for k, (lo, hi, kind) in SPACE.items():
        v = values[k]
        x.append((np.log(v) - np.log(lo)) / (np.log(hi) - np.log(lo)) if kind == "log" else (v - lo) / (hi - lo))
    return np.clip(np.array(x, dtype=float), 0, 1)

def spec_of(values):
    return "ref:" + ",".join(f"{k}={v}" for k, v in values.items())

print(spec_of(decode(encode(REF_VALUES))))

In [ ]:
def default_screen_set(phase0_csv=None, n=10):
    """Top contributors from the phase-0 run if available, else two scenarios per station."""
    for p in [phase0_csv, REPO / "notebooks" / "eval_ref" / "scores_ref.csv", REPO / "eval_ref" / "scores_ref.csv"]:
        if p and Path(p).exists():
            df = pd.read_csv(p).sort_values("score", ascending=False)
            print("screen set: top", n, "contributors from", p)
            return df.scenario.head(n).tolist()
    allsc = paths.training_scenarios()
    by_station = {}
    for s in allsc:
        by_station.setdefault(s.split("/")[1].rsplit("_", 1)[0], []).append(s)
    pick = [s for v in by_station.values() for s in v[::3][:2]]
    print("screen set: 2 per station (no phase-0 results found)")
    return pick

MODE = "shared"               # "shared" | "per_scenario"
SEARCH_STEPS = 2016           # 7 days per evaluation
GENERATIONS = 10
POPSIZE = 6
OUT = REPO / "results" / "search"

TUNE = default_screen_set(n=6)
HOLDOUT = [s for s in paths.training_scenarios() if s not in TUNE]
print("tune on:", TUNE)
cost = GENERATIONS * POPSIZE * len(TUNE)
print(f"budget: {cost} runs of {SEARCH_STEPS} steps")

In [ ]:
def evaluate(values_list, scenarios, steps=SEARCH_STEPS):
    """Mean score of each parameter set over the scenarios (runs in parallel, cached)."""
    specs = [spec_of(v) for v in values_list]
    ev.run_batch([(s, sp, steps) for sp in specs for s in scenarios], OUT, workers=WORKERS, verbose=False)
    df = ev.load_results(OUT, specs=set(specs), steps=steps)
    df = df[df.scenario.isin(scenarios)]
    means = df.groupby("spec").score.mean()
    return [float(means.get(sp, np.inf)) for sp in specs]


def search(scenarios, generations=GENERATIONS, popsize=POPSIZE, seed=0):
    """CMA-ES from the reference parameters; returns the log of every evaluation."""
    log = []
    x0 = encode(REF_VALUES)
    ref_score = evaluate([REF_VALUES], scenarios)[0]
    log.append({"gen": -1, **REF_VALUES, "score": ref_score})
    print(f"ref: {ref_score:.5f}")
    try:
        import cma
        es = cma.CMAEvolutionStrategy(x0, 0.2, {"popsize": popsize, "bounds": [0, 1], "seed": seed + 1, "verbose": -9})
        for g in range(generations):
            xs = es.ask()
            vals = [decode(x) for x in xs]
            scores = evaluate(vals, scenarios)
            es.tell(xs, scores)
            log += [{"gen": g, **v, "score": s} for v, s in zip(vals, scores)]
            print(f"gen {g:2d}: best {min(scores):.5f}  mean {np.mean(scores):.5f}  (ref {ref_score:.5f})", flush=True)
    except ImportError:
        print("cma not installed (pip install cma): random search around ref")
        rng = np.random.default_rng(seed)
        for g in range(generations):
            xs = [np.clip(x0 + rng.normal(0, 0.2, len(x0)), 0, 1) for _ in range(popsize)]
            vals = [decode(x) for x in xs]
            scores = evaluate(vals, scenarios)
            log += [{"gen": g, **v, "score": s} for v, s in zip(vals, scores)]
            print(f"gen {g:2d}: best {min(scores):.5f}  (ref {ref_score:.5f})", flush=True)
    return pd.DataFrame(log)

## Tìm

In [ ]:
if MODE == "shared":
    logs = {"shared": search(TUNE)}
else:
    logs = {s: search([s]) for s in TUNE}
for key, log in logs.items():
    log.to_csv(OUT / f"log_{key.replace('/', '__')}.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
for key, log in logs.items():
    g = log[log.gen >= 0].groupby("gen").score.min().cummin()
    ax.plot(g.index, g.values, marker="o", label=key)
    ax.axhline(log.loc[log.gen == -1, "score"].iloc[0], ls="--", lw=1, color=ax.lines[-1].get_color())
ax.set_xlabel("thế hệ"); ax.set_ylabel("điểm tốt nhất (thấp = tốt)"); ax.set_title("Hội tụ (nét đứt = ref)")
ax.legend(fontsize=8); plt.show()

## Kiểm tra trên kịch bản chưa dùng để tune

Tune trên 7 ngày dễ khớp quá mức; chỉ tin bộ tham số nếu nó thắng ref trên các kịch bản giữ lại, chạy đủ 30 ngày.

In [ ]:
CHECK_TOP = 3
FULL_STEPS = 8640
log = logs["shared"] if MODE == "shared" else pd.concat(logs.values())
top = (log[log.gen >= 0].sort_values("score").drop_duplicates(subset=list(SPACE)).head(CHECK_TOP))
top_vals = [{k: (int(r[k]) if SPACE[k][2] == "int" or k == "HOLD_STEPS" else r[k]) for k in SPACE} for _, r in top.iterrows()]
specs = ["ref"] + [spec_of(v) for v in top_vals]
ev.run_batch([(s, sp, FULL_STEPS) for sp in specs for s in HOLDOUT], OUT, workers=WORKERS)
full = ev.load_results(OUT, specs=set(specs), steps=FULL_STEPS)
full = full[full.scenario.isin(HOLDOUT)]
res = ev.compare(full, "ref")
display(res.round(5))
res.to_csv(OUT / "holdout_check.csv")
best_spec = res.index[0]
print("best on hold-out:", best_spec)

**Đọc kết quả:** nếu bộ tốt nhất không thắng `ref` trên kịch bản giữ lại, tham số không phải nút thắt → chuyển sang sửa logic (notebook 01) hoặc RL.
Với `MODE = "per_scenario"`, khoảng cách giữa điểm tune riêng và tune chung chính là "cổng" của giai đoạn 1 trong kế hoạch.